# Glucast — Ablación de variables de Garmin
## TFM — Máster Universitario en Inteligencia Artificial (UNIR) · Pablo Daniel Figueroa

**Pregunta:** ¿mejora la predicción si el LSTM propio recibe, además de las 8 variables comunes, el contexto diario del Garmin Instinct (estrés, frecuencia cardíaca en reposo, pasos y minutos de actividad)?

**Diseño**
- Mismos datos de glucosa y tratamientos que la comparativa (se leen los CSV crudos de una carpeta `resultados_…`, sin volver a descargar Nightscout) y **mismo conjunto de prueba** (último 15 % cronológico).
- Dos variantes con la misma arquitectura: **LSTM propio (8 variables)** y **LSTM propio + Garmin (12 variables)**. Cada una se entrena con 3 semillas y se promedian sus predicciones, para que la diferencia no dependa del azar del entrenamiento.
- **Sin fuga de información:** el estrés, la FC en reposo, los pasos y los minutos de actividad son los del **día anterior** (el día en curso todavía no terminó).
- **Sueño:** desactivado por defecto (`USAR_SUENO = False`). En la exportación actual casi todas las noches figuran con la ventana por defecto de 8 h y sin fases de sueño, es decir, el reloj no lo está registrando. Si se activa el registro de sueño en el Instinct, poner `USAR_SUENO = True`: se usa la última noche **ya terminada** en cada instante.
- Significación: **bootstrap pareado por días** (como en el capítulo 6).
- El dataset unificado no puede usar Garmin (OhioT1DM no tiene esas variables), por eso la ablación se hace sobre el modelo propio.

**Requisito:** `garmin_datos_consolidados.csv` actualizado (que cubra el período de prueba, septiembre–octubre de 2026) en la carpeta ENTRENAMIENTO de Drive. Se genera con `procesar_garmin.py` a partir de la exportación de Garmin Connect.

> *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. Tarda unos 15–25 min.

## 1. Librerías y configuración

In [ ]:
import os, time, glob, warnings
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
warnings.filterwarnings('ignore')

CARPETA_DRIVE = '/content/drive/MyDrive/MAESTRIA EN IA/TFM/ENTRENAMIENTO'
CARPETA_BASE  = None   # None = la carpeta resultados_… más reciente con los CSV crudos de Nightscout
ARCHIVO_GARMIN = 'garmin_datos_consolidados.csv'
SUENO_EN_UTC  = True   # procesar_garmin.py guarda inicio/fin del sueño en hora GMT
ZONA_HORARIA  = 'America/Argentina/Buenos_Aires'
COBERTURA_MINIMA_TEST = 0.80   # fracción de días de prueba con datos de Garmin para que el experimento sea válido

VENTANA, HORIZONTES, SPLIT = 24, [6, 12], (0.70, 0.15, 0.15)
EPOCHS, LR_INICIAL, BOOTSTRAP_B = 100, 1e-3, 1000
SEMILLAS = [42, 7, 123]
DIA_PERIODOS, ABSORCION_CARBS, ESCALA_DELTA = 24, 36, 100

BASE = ['glucosa', 'tendencia', 'bolo_u', 'iob', 'carbs', 'cob', 'hora_sin', 'hora_cos']
USAR_SUENO = False   # en la exportación de 2022–2026 el sueño figura casi siempre como la ventana por defecto (8 h) y sin fases: no aporta información real
GARMIN = ['estres_ant', 'fc_reposo_ant', 'pasos_ant', 'actividad_ant'] + (['sueno_h'] if USAR_SUENO else [])
ESCALAS = {'glucosa': 400, 'tendencia': 50, 'bolo_u': 10, 'iob': 10, 'carbs': 100, 'cob': 100, 'hora_sin': 1, 'hora_cos': 1,
           'estres_ant': 100, 'fc_reposo_ant': 100, 'pasos_ant': 20000, 'actividad_ant': 120, 'sueno_h': 10}
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'NO')

## 2. Drive y datos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
if CARPETA_BASE is None:
    cands = sorted(c for c in glob.glob(os.path.join(CARPETA_DRIVE, 'resultados_*')) if os.path.exists(os.path.join(c, 'nightscout_sgv_crudo.csv')))
    CARPETA_BASE = cands[-1]
CARPETA_RES = os.path.join(CARPETA_BASE, 'ablacion_garmin_' + datetime.now().strftime('%Y%m%d_%H%M'))
os.makedirs(CARPETA_RES, exist_ok=True)
df_sgv = pd.read_csv(os.path.join(CARPETA_BASE, 'nightscout_sgv_crudo.csv'))
df_trt = pd.read_csv(os.path.join(CARPETA_BASE, 'nightscout_tratamientos_crudo.csv'))
garmin = pd.read_csv(os.path.join(CARPETA_DRIVE, ARCHIVO_GARMIN))
print('Base:', CARPETA_BASE, '| lecturas:', len(df_sgv), '| Garmin:', garmin['fecha'].min(), '→', garmin['fecha'].max())

## 3. Preprocesamiento (idéntico a la comparativa) y variables de Garmin sin fuga

In [ ]:
def a_local(ts): return ts.dt.tz_convert(ZONA_HORARIA).dt.tz_localize(None)

def construir_base(gluc, bolos, carbs):
    g = gluc.groupby(level=0).mean().resample('5min').mean().interpolate(method='linear', limit=3, limit_area='inside')
    df = pd.DataFrame({'glucosa': g})
    df['bolo_u'] = bolos.resample('5min').sum().reindex(df.index).fillna(0) if len(bolos) else 0.0
    df['carbs'] = carbs.resample('5min').sum().reindex(df.index).fillna(0) if len(carbs) else 0.0
    df['iob'] = np.convolve(df['bolo_u'].values, np.maximum(0, 1 - np.arange(DIA_PERIODOS) / DIA_PERIODOS))[:len(df)]
    df['cob'] = np.convolve(df['carbs'].values, np.maximum(0, 1 - np.arange(ABSORCION_CARBS) / ABSORCION_CARBS))[:len(df)]
    df['tendencia'] = df['glucosa'].diff()
    hora = df.index.hour + df.index.minute / 60
    df['hora_sin'], df['hora_cos'] = np.sin(2 * np.pi * hora / 24), np.cos(2 * np.pi * hora / 24)
    return df

df_sgv['ts'] = pd.to_datetime(df_sgv['ts'], utc=True, format='ISO8601')
s_g = pd.Series(df_sgv['glucosa'].astype(float).values, index=a_local(df_sgv['ts'])).sort_index()
s_g = s_g[(s_g >= 39) & (s_g <= 401)]
df_trt['ts'] = pd.to_datetime(df_trt['ts'], utc=True, format='ISO8601')
tl = df_trt.assign(ts=a_local(df_trt['ts'])).set_index('ts').sort_index()
df = construir_base(s_g, tl['bolo_u'][tl['bolo_u'] > 0], tl['carbs'][tl['carbs'] > 0])

# --- Garmin: valores del DÍA ANTERIOR (no se usa información del propio día, que todavía no terminó)
g = garmin.copy(); g['fecha'] = pd.to_datetime(g['fecha']).dt.normalize()
g.loc[g['estres_promedio'] < 0, 'estres_promedio'] = np.nan   # Garmin usa -1/-2 cuando no pudo medir el estrés
g['actividad'] = g['minutos_intensidad_moderada'].fillna(0) + 2 * g['minutos_intensidad_vigorosa'].fillna(0)
diario = g.set_index('fecha')[['estres_promedio', 'fc_reposo', 'pasos_totales', 'actividad']]
diario.columns = ['estres_ant', 'fc_reposo_ant', 'pasos_ant', 'actividad_ant']
diario.index = diario.index + pd.Timedelta(days=1)        # el dato del día D se usa en el día D+1
dia = df.index.normalize()
for c in diario.columns:
    df[c] = diario[c].reindex(dia).values

if USAR_SUENO:
    # --- Sueño: el de la última noche que YA terminó en cada instante
    sn = g.dropna(subset=['sueno_fin']).copy()
    sn['fin'] = pd.to_datetime(sn['sueno_fin'])
    if SUENO_EN_UTC:
        sn['fin'] = sn['fin'].dt.tz_localize('UTC').dt.tz_convert(ZONA_HORARIA).dt.tz_localize(None)
    sn['sueno_h'] = (sn['sueno_profundo_seg'].fillna(0) + sn['sueno_liviano_seg'].fillna(0) + sn.get('sueno_rem_seg', 0)) / 3600
    sn = sn[sn['sueno_h'] > 0].sort_values('fin')[['fin', 'sueno_h']]
    aux = pd.merge_asof(pd.DataFrame({'t': df.index}), sn, left_on='t', right_on='fin', direction='backward', tolerance=pd.Timedelta(hours=36))
    df['sueno_h'] = aux['sueno_h'].values

t2 = df.index[int(len(df) * (SPLIT[0] + SPLIT[1]))]
dias_test = df.index[df.index >= t2].normalize().unique()
cob = {c: df.loc[df.index >= t2, c].notna().groupby(df.index[df.index >= t2].normalize()).any().mean() for c in GARMIN}
print('Cobertura de Garmin en los días de prueba:', {k: f'{v:.0%}' for k, v in cob.items()})
if min(cob.values()) < COBERTURA_MINIMA_TEST:
    raise SystemExit('⚠ El archivo de Garmin no cubre el período de prueba. Exportá los datos actualizados de Garmin Connect, '
                     'corré procesar_garmin.py y volvé a ejecutar.')

# huecos de Garmin (días sin reloj): se rellenan con la mediana de ENTRENAMIENTO
t1 = df.index[int(len(df) * SPLIT[0])]
for c in GARMIN:
    df[c] = df[c].fillna(df.loc[df.index < t1, c].median())
print(df[GARMIN].describe().round(1))

## 4. Ventanas, partición y modelo (iguales a la comparativa)

In [ ]:
def ventanas(df, H, feats):
    gl = df['glucosa'].values
    F = np.stack([(df[c].fillna(0).values / ESCALAS[c]) for c in feats], axis=1).astype('float32')
    n = len(df) - VENTANA - H + 1
    valido = ~np.isnan(gl); acum = np.concatenate([[0], np.cumsum(~valido)]); ini = np.arange(n)
    ok = (acum[ini + VENTANA] - acum[np.maximum(ini - 1, 0)] == 0) & (ini >= 1) & valido[ini + VENTANA - 1 + H]
    ini = ini[ok]
    X = np.lib.stride_tricks.sliding_window_view(F, (VENTANA, F.shape[1]))[:, 0][ini]
    ga, gf = gl[ini + VENTANA - 1], gl[ini + VENTANA - 1 + H]
    return X.astype('float32'), ((gf - ga) / ESCALA_DELTA).astype('float32'), ga, gf, pd.DatetimeIndex(df.index[ini + VENTANA - 1 + H])

def dividir(H, feats):
    X, y, ga, gf, t = ventanas(df, H, feats)
    t1 = df.index[int(len(df) * SPLIT[0])]; t2 = df.index[int(len(df) * (SPLIT[0] + SPLIT[1]))]
    m = pd.Timedelta(minutes=5 * (VENTANA + H))
    sel = {'train': t < t1, 'val': (t >= t1 + m) & (t < t2), 'test': t >= t2 + m}
    return {k: (X[s], y[s], ga[s], gf[s], t[s]) for k, s in sel.items()}

def construir(nf):
    m = Sequential([Input((VENTANA, nf)), LSTM(64, return_sequences=True), BatchNormalization(), Dropout(0.2),
                    LSTM(32), BatchNormalization(), Dropout(0.2), Dense(16, activation='relu'), Dense(1)])
    m.compile(optimizer=Adam(LR_INICIAL), loss='mse'); return m

def entrenar_y_predecir(D, nombre):
    preds = []
    for s in SEMILLAS:
        tf.keras.utils.set_random_seed(s)
        m = construir(D['train'][0].shape[2]); t0 = time.time()
        h = m.fit(D['train'][0], D['train'][1], validation_data=D['val'][:2], epochs=EPOCHS, batch_size=128, verbose=0,
                  callbacks=[EarlyStopping(patience=12, restore_best_weights=True), ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6)])
        preds.append(D['test'][2] + m.predict(D['test'][0], batch_size=2048, verbose=0).ravel() * ESCALA_DELTA)
        print(f"  {nombre:24s} semilla {s:3d}: {len(h.history['loss']):3d} épocas, {time.time()-t0:4.0f}s, RMSE test {np.sqrt(np.mean((preds[-1]-D['test'][3])**2)):.2f}")
    return np.mean(preds, axis=0), [np.sqrt(np.mean((p - D['test'][3]) ** 2)) for p in preds]

def bootstrap_pareado(ref, pa, pb, t, B=BOOTSTRAP_B):
    dias = pd.DatetimeIndex(t).normalize(); _, inv = np.unique(dias, return_inverse=True)
    sa, sb, n = np.bincount(inv, (pa - ref) ** 2), np.bincount(inv, (pb - ref) ** 2), np.bincount(inv)
    idx = np.random.default_rng(42).integers(0, len(n), (B, len(n)))
    d = np.sqrt(sa[idx].sum(1) / n[idx].sum(1)) - np.sqrt(sb[idx].sum(1) / n[idx].sum(1))
    return np.sqrt(np.mean((pa - ref) ** 2)) - np.sqrt(np.mean((pb - ref) ** 2)), np.percentile(d, 2.5), np.percentile(d, 97.5)

## 5. Experimento

In [ ]:
filas, sem = [], []
for H in HORIZONTES:
    print(f'\n=== Horizonte {H*5} min ===')
    Db, Dg = dividir(H, BASE), dividir(H, BASE + GARMIN)
    assert np.array_equal(Db['test'][4], Dg['test'][4]), 'los conjuntos de prueba deben ser idénticos'
    pb, rb = entrenar_y_predecir(Db, 'LSTM propio')
    pg, rg = entrenar_y_predecir(Dg, 'LSTM propio + Garmin')
    ref, t = Db['test'][3], Db['test'][4]
    d, lo, hi = bootstrap_pareado(ref, pg, pb, t)
    bajo = ref < 80
    for nombre, p, rs in (('LSTM propio', pb, rb), ('LSTM propio + Garmin', pg, rg)):
        filas.append({'Horizonte_min': H * 5, 'Modelo': nombre, 'RMSE': np.sqrt(np.mean((p - ref) ** 2)), 'MAE': np.mean(np.abs(p - ref)),
                      'RMSE glucosa < 80': np.sqrt(np.mean((p[bajo] - ref[bajo]) ** 2)), 'RMSE por semilla': ' / '.join(f'{x:.2f}' for x in rs), 'n_test': len(ref)})
    sem.append({'Horizonte_min': H * 5, 'ΔRMSE (Garmin − propio)': d, 'IC95': f'[{lo:.2f}, {hi:.2f}]',
                'Conclusión': 'Garmin mejora (significativo)' if hi < 0 else ('Garmin empeora (significativo)' if lo > 0 else 'sin diferencia significativa')})
res = pd.DataFrame(filas); sig = pd.DataFrame(sem)
res.to_csv(os.path.join(CARPETA_RES, 'tabla_ablacion_garmin.csv'), index=False)
sig.to_csv(os.path.join(CARPETA_RES, 'tabla_ablacion_garmin_bootstrap.csv'), index=False)
display(res.round(2)); display(sig.round(2))
print('Guardado en', CARPETA_RES)

## 6. Cómo leer el resultado
- **ΔRMSE negativo con IC95 entero por debajo de 0** → las variables de Garmin mejoran la predicción de forma significativa: se incorporan al modelo personal y se documenta en el capítulo 6.
- **IC95 que contiene el 0** → no hay evidencia de mejora con resolución diaria; se mantiene el modelo de 8 variables y se refuerza la línea futura de la Garmin Health API (datos minuto a minuto).
- La columna *RMSE por semilla* muestra cuánto varía cada variante entre entrenamientos: si la diferencia entre modelos es menor que esa variación, no conviene sacar conclusiones.

Pasale a Claude las dos tablas (o avisale que terminó y las lee desde Drive) para sumarlas al borrador.